# 03 · 第一个神经网络

目标很简单：**让神经网络学会 $y = 2x + 1$**。

听起来小儿科，但麻雀虽小五脏俱全——训练一个能识别猫的大模型，用的是一模一样的流程。

学完这一节，你就掌握了 PyTorch 的骨架。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)     # 固定随机种子，保证每次结果一样，方便对照
print('环境就绪')

## 1. 造数据

真实的机器学习里，数据是从文件里读的。这里我们**自己造**一批：随机取 200 个 $x$，按 $y = 2x + 1$ 算出 $y$。

然后让模型只看 $x$ 和 $y$，自己摸索出这条规律。

In [ ]:
X = torch.randn(200, 1)      # 200 个样本，每个是 1 维的 x
y = 2 * X + 1                # 真实规律

print('输入 X 的形状:', tuple(X.shape), '  → 200 行 1 列')
print('标签 y 的形状:', tuple(y.shape))
print()
print('前 5 条数据:')
for i in range(5):
    print(f'  x = {X[i].item():7.4f}   ->   y = {y[i].item():7.4f}')

## 2. 搭模型

一个「线性层 + 激活函数 + 线性层」的小网络。

几层就够——因为这个任务太简单了。**以后遇到难任务，就是在这里加宽加深。**

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 16),    # 输入 1 维 → 16 维
    nn.ReLU(),           # 激活函数，给网络加「非线性」
    nn.Linear(16, 1),    # 16 维 → 输出 1 维
)

print(model)
print()
print('模型里的参数（现在还是随机初始化的瞎猜值）:')
for name, p in model.named_parameters():
    print(f'  {name:16s} 形状 {tuple(p.shape)}')

In [ ]:
# 训练之前先看看模型瞎猜得有多离谱
with torch.no_grad():
    pred = model(X)

print('训练前，拿模型去预测:')
for i in range(3):
    print(f'  x={X[i].item():7.4f}   模型预测={pred[i].item():9.4f}   真实值={y[i].item():7.4f}')

## 3. 损失函数 + 优化器

- **损失函数**：衡量「预测值和真实值差多少」，差得越多损失越大
- **优化器**：根据损失的梯度，自动调整模型参数

- `MSELoss` = 均方误差，回归任务最常用
- `Adam` = 一个聪明好用的优化器，新手默认选它就行

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print('损失函数:', loss_fn)
print('优化器  :', optimizer)
print('学习率  : 0.05')

## 4. 训练循环 ⭐

**这是整份笔记最重要的一段代码。** 所有 PyTorch 训练都是这五步：

```
1. 前向传播  pred = model(X)        用当前参数算预测
2. 计算损失  loss = loss_fn(...)    看差多少
3. 梯度清零  optimizer.zero_grad()  清掉上一轮的旧梯度
4. 反向传播  loss.backward()        自动求所有参数的梯度
5. 更新参数  optimizer.step()       按梯度改进一点点
```

**顺序不能乱，一步都不能少。**

In [ ]:
print('epoch      loss')
print('-' * 24)

for epoch in range(301):
    pred = model(X)                    # 1. 前向传播
    loss = loss_fn(pred, y)            # 2. 计算损失
    optimizer.zero_grad()              # 3. 梯度清零
    loss.backward()                    # 4. 反向传播
    optimizer.step()                   # 5. 更新参数
    
    if epoch % 50 == 0:
        print(f'{epoch:5d}   {loss.item():10.6f}')

print('-' * 24)
print('训练完成')

损失从一个大数一路降到接近 0 —— 模型学会了。

## 5. 验证：它真的学会了吗？

拿几个模型**从没见过**的 $x$ 去问它。

In [ ]:
model.eval()                        # 切换到评估模式
with torch.no_grad():               # 不需要梯度
    test_x = torch.tensor([[0.0], [1.0], [2.0], [5.0], [-3.0]])
    pred = model(test_x)

print('   x      模型预测      真实值(2x+1)     误差')
print('-' * 48)
for i in range(len(test_x)):
    xi = test_x[i].item()
    true_y = 2 * xi + 1
    print(f'{xi:6.2f}   {pred[i].item():10.4f}   {true_y:12.4f}   {abs(pred[i].item()-true_y):8.5f}')

误差极小，说明模型确实学到了 $y = 2x + 1$ 这条规律，而不是死记硬背那 200 个点。

这就是深度学习：**从数据里找规律，然后用来预测没见过的情况。**

## 动手练习 ⭐ 这一节的重点

### 练习 1：改学习率
把 `lr` 改成 `0.001`（太小）和 `5.0`（太大）各跑一次，观察损失曲线有什么不同。

### 练习 2：加宽网络
把 `nn.Linear(1, 16)` 的 16 改成 3 或 128，看效果有没有区别。

### 练习 3：⚠️ 陷阱题
把数据改成像这样：
```python
y = X ** 2
```
然后**不做任何其他改动**，直接训练。

你会发现模型怎么都学不好，损失降不下去。

**想想为什么？** 提示：仔细看模型里的 `nn.ReLU()` —— 如果把激活函数去掉，会发生什么？

> 这道题背后是深度学习的核心思想之一：**没有非线性的网络，再深也只是个线性模型。**

In [ ]:
# 在这里做练习
# 提示：可以复制上面的「造数据」和「训练循环」两格，改一改再运行


---

## 这一节要记住的

**训练的五个步骤**（背下来，以后所有模型都用它）：

| 步骤 | 代码 |
|---|---|
| 1. 前向传播 | `pred = model(X)` |
| 2. 计算损失 | `loss = loss_fn(pred, y)` |
| 3. 梯度清零 | `optimizer.zero_grad()` |
| 4. 反向传播 | `loss.backward()` |
| 5. 更新参数 | `optimizer.step()` |

其他要点：
- `nn.Sequential` 像搭积木一样堆层
- 激活函数（如 `ReLU`）负责引入非线性，**不能去掉**
- 预测时用 `model.eval()` + `with torch.no_grad():`
- 学习率太大不收敛、太小太慢

---

**下一步学什么？** 看同目录下的《学习路线.md》。